# Question 3: Association Rule Mining
Dataset: `mobile_price.csv`, filtered to `price_range == 1`  
Features: `ram`, `int_memory`, `px_width`, `battery_power`

In [1]:
#!pip install mlxtend --user
#!pip install "numpy<2" --user
#!pip install jinja2


In [2]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules

c:\Users\janas\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\janas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Data Preparation: Filter, Categorize, Build Transactions

In [3]:
df = pd.read_csv('mobile_price.csv')

# Filter: only price_range == 1
df_filtered = df[df['price_range'] == 1].copy().reset_index(drop=True)
print(f'Samples with price_range=1: {len(df_filtered)}')

features = ['ram', 'int_memory', 'px_width', 'battery_power']

Samples with price_range=1: 500


In [4]:
def categorize_feature(series):
    """Divide values into low/medium/high using 3:4:3 ratio on value range."""
    min_val = series.min()
    max_val = series.max()
    range_val = max_val - min_val

    low_thresh  = min_val + 0.3 * range_val   # bottom 30%
    high_thresh = min_val + 0.7 * range_val   # top 30% starts here

    print(f'  {series.name:<15} min={min_val}, max={max_val}, '
          f'low<{low_thresh:.1f}, medium<{high_thresh:.1f}, high>={high_thresh:.1f}')

    def label(val):
        if val < low_thresh:
            return 'low'
        elif val < high_thresh:
            return 'medium'
        else:
            return 'high'

    return series.apply(label)

print('Categorization thresholds:')
df_cat = pd.DataFrame()
for feat in features:
    df_cat[feat] = categorize_feature(df_filtered[feat])

print('\nFirst 5 categorized rows:')
print(df_cat.head())

Categorization thresholds:
  ram             min=387, max=2811, low<1114.2, medium<2083.8, high>=2083.8
  int_memory      min=2, max=64, low<20.6, medium<45.4, high>=45.4
  px_width        min=500, max=1998, low<949.4, medium<1548.6, high>=1548.6
  battery_power   min=501, max=1996, low<949.5, medium<1547.5, high>=1547.5

First 5 categorized rows:
      ram int_memory px_width battery_power
0    high        low      low           low
1  medium     medium   medium          high
2     low     medium     high          high
3  medium     medium      low          high
4  medium       high      low        medium


In [5]:
# Show example transactions
print('Example transactions (first 5):')
for i in range(5):
    transaction = [f'{feat}_{df_cat.loc[i, feat]}' for feat in features]
    print(f'  Row {i}: {transaction}')

Example transactions (first 5):
  Row 0: ['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low']
  Row 1: ['ram_medium', 'int_memory_medium', 'px_width_medium', 'battery_power_high']
  Row 2: ['ram_low', 'int_memory_medium', 'px_width_high', 'battery_power_high']
  Row 3: ['ram_medium', 'int_memory_medium', 'px_width_low', 'battery_power_high']
  Row 4: ['ram_medium', 'int_memory_high', 'px_width_low', 'battery_power_medium']


In [6]:
# Build one-hot encoded DataFrame required by mlxtend
# Each column is one item, e.g. 'ram_high', value is True/False
te_df = pd.DataFrame()
for feat in features:
    for level in ['low', 'medium', 'high']:
        item_name = f'{feat}_{level}'
        te_df[item_name] = (df_cat[feat] == level)

print(f'One-hot encoded shape: {te_df.shape}  ({te_df.shape[1]} items, {te_df.shape[0]} transactions)')
te_df.head()

One-hot encoded shape: (500, 12)  (12 items, 500 transactions)


,ram_low,ram_medium,ram_high,int_memory_low,int_memory_medium,int_memory_high,px_width_low,px_width_medium,px_width_high,battery_power_low,battery_power_medium,battery_power_high
0,False,False,True,True,False,False,True,False,False,True,False,False
1,False,True,False,False,True,False,False,True,False,False,False,True
2,True,False,False,False,True,False,False,False,True,False,False,True
3,False,True,False,False,True,False,True,False,False,False,False,True
4,False,True,False,False,False,True,True,False,False,False,True,False


## 3a. Frequent Patterns with support ≥ 0.3 using FP-growth

In [7]:
freq_itemsets = fpgrowth(te_df, min_support=0.3, use_colnames=True)
freq_itemsets = freq_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

# Add itemset length for clarity
freq_itemsets['length'] = freq_itemsets['itemsets'].apply(len)

print(f'Total frequent itemsets found: {len(freq_itemsets)}')
freq_itemsets

Total frequent itemsets found: 8


,support,itemsets,length
0,0.682,frozenset({ram_medium}),1
1,0.416,frozenset({px_width_medium}),1
2,0.414,frozenset({battery_power_medium}),1
3,0.412,frozenset({int_memory_medium}),1
4,0.318,"frozenset({ram_medium, battery_power_medium})",2
5,0.316,frozenset({int_memory_low}),1
6,0.308,frozenset({battery_power_low}),1
7,0.306,"frozenset({ram_medium, px_width_medium})",2


## 3b. Association Rules with support ≥ 0.3, confidence ≥ 0.4, lift ≥ 0.8

In [8]:
rules = association_rules(freq_itemsets, metric='confidence', min_threshold=0.4)

# Apply remaining filters
rules = rules[
    (rules['support']    >= 0.3) &
    (rules['lift']       >= 0.8)
].reset_index(drop=True)

rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Total association rules found: {len(rules)}')
rules.round(4)

Total association rules found: 4


,antecedents,consequents,support,confidence,lift
0,frozenset({ram_medium}),frozenset({battery_power_medium}),0.318,0.4663,1.1263
1,frozenset({battery_power_medium}),frozenset({ram_medium}),0.318,0.7681,1.1263
2,frozenset({ram_medium}),frozenset({px_width_medium}),0.306,0.4487,1.0786
3,frozenset({px_width_medium}),frozenset({ram_medium}),0.306,0.7356,1.0786
